<a href="https://colab.research.google.com/github/sartha-k/gan-mnist/blob/main/gan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
class discriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.feature = nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*7*7,1024),
        nn.ReLU(),
        nn.Linear(1024,1),
        nn.Sigmoid()
    )

  def forward(self,out):
    out = self.feature(out)
    out = self.classifier(out)
    return out





In [ ]:
class generator(nn.Module):
  def __init__(self):
   super().__init__()
   self.projection = nn.Sequential(
       nn.Linear(100,128*7*7), #100->128*7*7
       nn.ReLU(),
       nn.Unflatten(1,(128,7,7))
   )
   self.feature = nn.Sequential(
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 7→14
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),   # 14→28
    nn.Tanh()
   )

  def forward(self,out):
   out = self.projection(out)
   out = self.feature(out)
   return out



In [ ]:
g = generator().to(device)
d = discriminator().to(device)
noise = torch.randn(32,100).to(device)
fake = g(noise)
print(fake.shape)
out = d(fake)
print(out.shape)

In [ ]:
lr = 0.0002
epochs=150
dataloader = DataLoader(train_data,batch_size=32,shuffle=True)
loss = nn.BCELoss()
optimiser_d = torch.optim.Adam(d.parameters(),lr)
optimiser_g = torch.optim.Adam(g.parameters(),lr)


In [ ]:
for epoch in range(epochs):
    for batch_idx, (real_images, _) in enumerate(dataloader):

        # Step 1: Train Discriminator

        # - get real images, label them 1
        images = real_images.to(device)
        real_labels = torch.ones(images.size(0), 1).to(device) * 0.9 # smooth labeling

        # - generate fake images, label them 0
        noise = torch.randn(images.size(0), 100).to(device)
        fake_labels = torch.zeros(noise.size(0), 1).to(device)

        # - compute discriminator loss on both
        loss_real = loss(d(images),real_labels)
        fake_images = g(noise)
        loss_fake = loss(d(fake_images.detach()),fake_labels)
        loss_d = loss_real + loss_fake

        # - update discriminator weights
        optimiser_d.zero_grad()
        loss_d.backward()
        optimiser_d.step()

        # Step 2: Train Generator

        # - generate fake images
        noise = torch.randn(images.size(0), 100).to(device)

        # - pass through discriminator
        fake_images = g(noise)

        # - loss = how well did it fool discriminator
        loss_g=-torch.mean(torch.log(d(fake_images)+1e-8))

        # - update generator weights
        optimiser_g.zero_grad()
        loss_g.backward()
        optimiser_g.step()
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch}/{epochs}] Batch {batch_idx} | Loss D: {loss_d.item():.4f} | Loss G: {loss_g.item():.4f}")

Epoch [0/150] Batch 0 | Loss D: 1.3665 | Loss G: 7.1599
Epoch [0/150] Batch 100 | Loss D: 0.3318 | Loss G: 7.3436
Epoch [0/150] Batch 200 | Loss D: 0.4006 | Loss G: 4.4132
Epoch [0/150] Batch 300 | Loss D: 0.4356 | Loss G: 3.2333
Epoch [0/150] Batch 400 | Loss D: 0.3631 | Loss G: 5.1069
Epoch [0/150] Batch 500 | Loss D: 0.3651 | Loss G: 6.3979
Epoch [0/150] Batch 600 | Loss D: 0.3431 | Loss G: 4.9976
Epoch [0/150] Batch 700 | Loss D: 0.3718 | Loss G: 5.2051
Epoch [0/150] Batch 800 | Loss D: 0.4309 | Loss G: 7.3563
Epoch [0/150] Batch 900 | Loss D: 0.3700 | Loss G: 6.4807
Epoch [0/150] Batch 1000 | Loss D: 0.3441 | Loss G: 8.7817
Epoch [0/150] Batch 1100 | Loss D: 0.3459 | Loss G: 4.6336
Epoch [0/150] Batch 1200 | Loss D: 0.3475 | Loss G: 9.0098
Epoch [0/150] Batch 1300 | Loss D: 0.3571 | Loss G: 7.1151
Epoch [0/150] Batch 1400 | Loss D: 0.3515 | Loss G: 6.6988
Epoch [0/150] Batch 1500 | Loss D: 0.3397 | Loss G: 6.9304
Epoch [0/150] Batch 1600 | Loss D: 0.4216 | Loss G: 5.4678
Epoch [0/

In [1]:
import matplotlib.pyplot as plt

g.eval()
with torch.no_grad():
    noise = torch.randn(16, 100).to(device)
    fake = g(noise)
    fake = fake.cpu().squeeze()

fig, axes = plt.subplots(4, 4, figsize=(6,6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(fake[i], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()


NameError: name 'g' is not defined